# Unit Conversions with unyt
In this tutorial we give the same examples as `astropy.ipynb` and `pint.ipynb`, but using [unyt](https://unyt.readthedocs.io/) as the unit engine. This makes it easier to compare where the three packages line up and where their APIs differ.

In [1]:
import math

import numpy as np
import unyt as u
import unyt.equivalencies as equivalencies
from unyt.dimensions import current_mks, length as length_dimension, magnetic_field

In unyt, a `unyt_quantity` or `unyt_array` is also the combination of a value and a unit. The most convenient way to create one is to multiply or divide a value by one of the built-in units. It works with scalars, sequences, and `numpy` arrays.

In [2]:
length = 1 * u.meter
length

unyt_quantity(1, 'm')

A quantity is composed of a value and units.

In [3]:
print(length.value)
print(length.units)

1
m


Most unyt units support prefixes, for example `k` for kilo.

In [4]:
length = 1 * u.km
length

unyt_quantity(1, 'km')

You can perform operations on quantities. Units remain and combine through the operations.

In [5]:
time = 2 * u.s

velocity = length / time
velocity

unyt_quantity(0.5, 'km/s')

unyt can convert to base units in a named unit system. This is the closest match to Astropy's `.si` and `.cgs` conveniences.

In [6]:
velocity.in_base("mks")

unyt_quantity(500., 'm/s')

In [7]:
velocity.in_base("cgs")

unyt_quantity(50000., 'cm/s')

It is also possible to specify explicitly which units you wish to convert to.

In [8]:
velocity.to(u.um / u.Gs)  # micrometer per gigasecond

unyt_quantity(5.e+17, 'μm/Gs')

unyt also works with `numpy` arrays.

In [9]:
a = np.array([1, 2, 3]) * u.m
b = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) * u.cm

display(a)
display(b)

unyt_array([1, 2, 3], 'm')

unyt_array([[1, 2, 3],
       [4, 5, 6],
       [7, 8, 9]], 'cm')

They keep the units even through complex operations.

In [10]:
out = np.matmul(b, a)
out.in_base("mks")

unyt_array([0.14, 0.32, 0.5 ], 'm**2')

## Magnetic units

Here we show how to use magnetic units with unyt. Often in the magnetism community, a mix of CGS and SI units are used. unyt includes some CGS magnetic units, but it does not include every magnetism convention used in the Astropy and Pint examples. Where needed, we define explicit SI-backed aliases below.

| Quantity Symbol | SI Units | cgs Units |
|-----------------|----------|-----------|
| Length $(x)$ | $10^{-2} \, \text{m}$ | $1 \, \text{cm}$ |
| Mass $(m)$ | $10^{-3} \, \text{kg}$ | $1 \, \text{g}$ |
| Force $(F)$ | $10^{-5} \, \text{N}$ | $1 \, \text{dyne}$ |
| Energy $(E)$ | $10^{-7} \, \text{J}$ | $1 \, \text{erg}$ |
| Magnetic Induction $(B)$ | $10^{-4} \, \text{T}$ | $1 \, \text{G}$ |
| Magnetic Field Strength $(H)$ | $\frac{10^{-4}}{\mu_0} \, \text{Am}^{-1}$ | $1 \, \text{Oe}$ |
| Magnetic Moment $(\mu)$ | $10^{-3} \, \text{JT}^{-1}$ or $\text{Am}^2$ | $1 \, \text{erg G}^{-1}$ or emu |
| Magnetization $(M)$ | $10^3 \, \text{Am}^{-1}$ or $\text{JT}^{-1} \, \text{m}^{-3}$ | $1 \, \text{Oe}$ or emu cm $^{-3}$ |
| Magnetic Susceptibility $(\chi)$ | $4\pi$ (dimensionless) | $1 \, \text{emu cm}^{-3} \, \text{Oe}^{-1}$ |
| Molar Susceptibility $(\chi_m)$ | $4\pi \times 10^{-6} \, \text{m}^3 \, \text{mol}^{-1}$ | $1 \, \text{emu mol}^{-1} \, \text{Oe}^{-1}$ |
| Mass Susceptibility $(\chi_g)$ | $4\pi \times 10^{-3} \, \text{m}^3 \, \text{kg}^{-1}$ | $1 \, \text{emu g}^{-1} \, \text{Oe}^{-1}$ |
| Magnetic Flux $(\phi)$ | $10^{-8} \, \text{Tm}^2$ or $\text{Wb}$ | $1 \, \text{G cm}^2$ or $\text{Mx}$ |
| Demagnetization Factor (N) | $0 < N < 1$ | $0 < N < 4\pi$ |

In [11]:
def ensure_unit(symbol, value, tex_repr=None):
    if not hasattr(u, symbol):
        u.define_unit(symbol, value, tex_repr=tex_repr)

mu_0 = 4 * math.pi * 1e-7 * u.N / u.A**2

ensure_unit("Oe", (1000 / (4 * math.pi), "A/m"), tex_repr=r"\rm{Oe}")
ensure_unit("oersted", (1.0, "Oe"), tex_repr=r"\rm{Oe}")
ensure_unit("emu_moment", 1e-3 * u.J / u.T, tex_repr=r"\rm{emu}")
ensure_unit("maxwell_si", 1e-8 * u.Wb, tex_repr=r"\rm{Mx}")

In [12]:
# Initialize quantities in cgs-style units
length = 1 * u.cm
mass = 1 * u.g
force = 1 * u.dyne
energy = 1 * u.erg
magnetic_induction = 1 * u.gauss
magnetic_field_strength = 1 * u.Oe
magnetic_moment = 1 * u.emu_moment
magnetization = 1 * u.emu_moment / u.cm**3
magnetic_flux = 1 * u.maxwell_si

print("| Quantity               | cgs Unit       | Converted SI Unit")
print("|------------------------|----------------|--------------------")
print(f"| Length                 | 1 cm           | {length.to(u.m):.2g}")
print(f"| Mass                   | 1 g            | {mass.to(u.kg):.2g}")
print(f"| Force                  | 1 dyne         | {force.to(u.N):.2g}")
print(f"| Energy                 | 1 erg          | {energy.to(u.J):.2g}")
print(f"| Magnetic Induction     | 1 G            | {magnetic_induction.to(u.T):.2g}")
print(f"| Magnetic Field Strength| 1 Oe           | {magnetic_field_strength.to(u.A / u.m):.2g}")
print(f"| Magnetic Moment        | 1 emu          | {magnetic_moment.to(u.J / u.T):.2g}")
print(f"| Magnetization          | 1 emu cm^-3    | {magnetization.to(u.A / u.m):.2g}")
print(f"| Magnetic Flux          | 1 Mx           | {magnetic_flux.to(u.Wb):.2g}")

| Quantity               | cgs Unit       | Converted SI Unit
|------------------------|----------------|--------------------
| Length                 | 1 cm           | 0.01 m
| Mass                   | 1 g            | 0.001 kg
| Force                  | 1 dyne         | 1e-05 N
| Energy                 | 1 erg          | 1e-07 J
| Magnetic Induction     | 1 G            | 0.0001 T
| Magnetic Field Strength| 1 Oe           | 80 A/m
| Magnetic Moment        | 1 emu          | 0.001 J/T
| Magnetization          | 1 emu cm^-3    | 1e+03 A/m
| Magnetic Flux          | 1 Mx           | 1e-08 Wb


There are aliases for certain units, but one important difference from Astropy and Pint is that `u.G` in unyt is the gravitational constant, not gauss. Use `u.gauss` or `u.Unit("gauss")` for the magnetic unit.

In [13]:
u.gauss == u.Unit("gauss")

True

unyt's built-in `Mx` and `maxwell` are CGS electromagnetic units, but they do not convert to SI `Wb` in compound magnetic workflows the same way Astropy and Pint do. The `maxwell_si` alias above is an explicit SI-backed representation of the conventional relation `1 Mx = 1e-8 Wb`.

## Equivalencies
unyt calls these `equivalences`. Built-in equivalences can be used by name, and custom equivalences can be registered by subclassing `unyt.equivalencies.Equivalence` with a `type_name`.

In [14]:
exchange = 4.15 * u.K
exchange.to(u.J, equivalence="thermal")

unyt_quantity(5.72969252e-23, 'J')

We can also convert magnetic field strength $(\mathbf{H})$ and magnetic flux density $(\mathbf{B})$ using the relationship:

$$
\mathbf{B} = \mu_r \mu_0 \mathbf{H}.
$$

unyt does not provide this magnetism-specific equivalence by default, but it can be registered. Without the custom equivalence, field strength cannot be converted directly to magnetic induction.

In [15]:
H = 1 * u.Oe
try:
    H.to(u.gauss)
except Exception as exc:
    print(type(exc).__name__)
    print(exc)

UnitConversionError
Cannot convert between 'Oe' (dim '(current_mks)/(length)') and 'G' (dim 'sqrt((mass))/(sqrt((length))*(time))').


In [16]:
class MagneticFluxFieldEquivalence(equivalencies.Equivalence):
    type_name = "magnetic_flux_field"
    _dims = (current_mks / length_dimension, magnetic_field)

    def _convert(self, quantity, new_dims, mu_r=1):
        if new_dims == magnetic_field:
            return (mu_r * mu_0 * quantity).to(u.T)
        if new_dims == current_mks / length_dimension:
            return (quantity.to(u.T) / (mu_r * mu_0)).to(u.A / u.m)
        raise equivalencies.InvalidUnitEquivalence(self, quantity.units, new_dims)

In [17]:
H.to(u.T, equivalence="magnetic_flux_field").to(u.gauss)

unyt_quantity(1., 'G')

The helper also accepts the relative permeability of the medium.

In [18]:
H.to(u.T, equivalence="magnetic_flux_field", mu_r=0.9).to(u.gauss)

unyt_quantity(0.9, 'G')

If we are doing multiple conversions, we can call the registered equivalence wherever the physical relationship is needed.

In [19]:
B = H.to(u.T, equivalence="magnetic_flux_field", mu_r=0.9)
display(B)

unyt_quantity(9.e-05, 'T')

We can now convert between the related SI and CGS magnetic quantities explicitly.

In [20]:
display(H)
display(H.to(u.T, equivalence="magnetic_flux_field").to(u.gauss))
display(H.to(u.T, equivalence="magnetic_flux_field"))
display(H.to(u.A / u.m))

unyt_quantity(1, 'Oe')

unyt_quantity(1., 'G')

unyt_quantity(0.0001, 'T')

unyt_quantity(79.57747155, 'A/m')

A custom equivalence can also be used to convert other common units in magnetism.

The magnetic moment can be converted to magnetic induction by providing the volume of the respective counting unit.

In the case of formula unit:

In [21]:
ensure_unit("mu_B", 9.2740100783e-24 * u.J / u.T, tex_repr=r"\mu_B")
ensure_unit("formula_unit", (1.0, "dimensionless"), tex_repr=r"\rm{f.u.}")
ensure_unit("f_u", (1.0, "formula_unit"), tex_repr=r"\rm{f.u.}")
ensure_unit("atom", (1.0, "dimensionless"), tex_repr=r"\rm{atom}")

In [22]:
class MomentInductionEquivalence(equivalencies.Equivalence):
    type_name = "moment_induction"
    _moment_dims = (u.mu_B / u.atom).units.dimensions
    _dims = (_moment_dims, magnetic_field)

    def _convert(self, quantity, new_dims, volume):
        volume = volume.to(u.m**3)
        if volume <= 0 * u.m**3:
            raise ValueError("Volume must be positive")
        if new_dims == magnetic_field:
            return (mu_0 * quantity / volume).to(u.T)
        if new_dims == self._moment_dims:
            return (quantity.to(u.T) * volume / mu_0).to(u.J / u.T)
        raise equivalencies.InvalidUnitEquivalence(self, quantity.units, new_dims)

In [23]:
magnetic_moment = 8 * u.mu_B / u.f_u

vol_per_fu = (5 * u.angstrom) ** 3

magnetic_moment.to(u.T, equivalence="moment_induction", volume=vol_per_fu)

unyt_quantity(0.74586015, 'T')

In [24]:
magnetic_induction = 1 * u.T

magnetic_induction.to(u.mu_B / u.f_u, equivalence="moment_induction", volume=vol_per_fu)

unyt_quantity(10.7258714, 'mu_B/f_u')

In [25]:
magnetic_moment = 2 * u.mu_B / u.atom

vol_per_atom = (3 * u.angstrom) ** 3

magnetic_moment.to(u.T, equivalence="moment_induction", volume=vol_per_atom)

unyt_quantity(0.86326406, 'T')

In [26]:
magnetic_induction = 1 * u.T

magnetic_induction.to(u.mu_B / u.atom, equivalence="moment_induction", volume=vol_per_atom)

unyt_quantity(2.31678822, 'mu_B/atom')

We can also convert using other forms of induction.

In [27]:
magnetic_induction = 10000 * u.gauss

magnetic_induction.to(u.T).to(
    u.mu_B / u.atom, equivalence="moment_induction", volume=vol_per_atom
)

unyt_quantity(2.31678822, 'mu_B/atom')

For field strength, we chain the registered equivalences: first convert field strength to induction, then convert induction to magnetic moment.

In [28]:
magnetic_field_strength = 10000 * u.Oe

magnetic_field_strength.to(u.T, equivalence="magnetic_flux_field").to(
    u.mu_B / u.atom,
    equivalence="moment_induction",
    volume=vol_per_atom,
)

unyt_quantity(2.31678822, 'mu_B/atom')

## A comment on emu

emu is not a unit in the conventional manner and can have many different meanings depending on context.

Within magnetism emu can represent the magnetic moment
$$
1\, \text{emu} = 1\, \frac{\text{erg}}{\text{G}}.
$$
However, it can sometimes have dimensions
$$
1\, \text{emu} = 1\, \text{cm}^3,
$$
such as representing emu in magnetic susceptibility if defined as (emu cm $^{-3}$). unyt does not define one universal `emu`; the notebook uses `emu_moment` as an explicit magnetic-moment convention.